# Riemann data pipeline — consolidated 00–05

Single Colab notebook combining the project stages in execution order.

Stages: 00 environment → 01 acquire → 02 describe → 03 unfold → 04 surrogates → 05 compare.

**Artifact rule:** an existing derived artifact is never overwritten. If it exists, its SHA-256 and byte size must match `data/manifest.json`; otherwise the pipeline stops with an error. If it does not exist, it is created, verified, and recorded in the manifest.

## 00 — Environment

In [ ]:
import sys
import json
import hashlib
from pathlib import Path

import numpy as np

print(sys.version)
print("numpy:", np.__version__)

REPO_BASE = Path("/content/nicht-riemann-data")
assert REPO_BASE.exists(), f"Repository not found: {REPO_BASE}"

%cd /content/nicht-riemann-data
!git status --short
!git branch --show-current


In [ ]:
!{sys.executable} -m pip install -e .

## Restart runtime

After the editable install, restart Colab before testing project imports.

In [ ]:
import os
os._exit(0)


In [ ]:
from nicht_riemann_data.transforms import spacings
from nicht_riemann_data.diagnostics import describe

print("Package import: OK")


## Integrity helpers

In [ ]:
MANIFEST_FILE = Path("data/manifest.json")
DATA_DIR = Path("data/raw")
DERIVED_DIR = Path("data/derived")
DATA_DIR.mkdir(parents=True, exist_ok=True)
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

assert MANIFEST_FILE.exists(), f"Missing manifest: {MANIFEST_FILE}"

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def load_manifest():
    with MANIFEST_FILE.open("r", encoding="utf-8") as f:
        return json.load(f)

manifest = load_manifest()
print("manifest:", MANIFEST_FILE)


## 01 — Acquire

An existing raw file is verified against the manifest and never silently replaced. A missing file is downloaded and then verified before use.

In [ ]:
import urllib.request

ODLYZKO_BASE = "https://www-users.cse.umn.edu/~odlyzko/zeta_tables"
DATASET = "zeros1"
URL = f"{ODLYZKO_BASE}/{DATASET}"
RAW_FILE = DATA_DIR / DATASET
raw_spec = manifest["datasets"]["odlyzko_zeros1"]
EXPECTED_RAW_SHA256 = raw_spec["sha256"]
EXPECTED_RAW_BYTES = raw_spec["raw_bytes"]

if RAW_FILE.exists():
    actual_bytes = RAW_FILE.stat().st_size
    actual_sha256 = sha256(RAW_FILE)
    assert actual_bytes == EXPECTED_RAW_BYTES, (
        f"Raw artifact size mismatch: {actual_bytes} != {EXPECTED_RAW_BYTES}"
    )
    assert actual_sha256 == EXPECTED_RAW_SHA256, (
        f"Raw artifact SHA-256 mismatch: {actual_sha256} != {EXPECTED_RAW_SHA256}"
    )
    print(f"Verified existing raw artifact: {RAW_FILE}")
else:
    print(f"Missing raw artifact; downloading: {URL}")
    urllib.request.urlretrieve(URL, RAW_FILE)
    actual_bytes = RAW_FILE.stat().st_size
    actual_sha256 = sha256(RAW_FILE)
    assert actual_bytes == EXPECTED_RAW_BYTES, (
        f"Downloaded raw artifact size mismatch: {actual_bytes} != {EXPECTED_RAW_BYTES}"
    )
    assert actual_sha256 == EXPECTED_RAW_SHA256, (
        f"Downloaded raw artifact SHA-256 mismatch: {actual_sha256} != {EXPECTED_RAW_SHA256}"
    )
    print(f"Downloaded and verified: {RAW_FILE}")

print("bytes:", actual_bytes)
print("SHA-256:", actual_sha256)


In [ ]:
gamma = np.loadtxt(RAW_FILE, dtype=np.float64)
assert gamma.ndim == 1
assert gamma.dtype == np.float64
assert np.all(np.isfinite(gamma))
assert np.all(np.diff(gamma) > 0)

delta = spacings(gamma)
assert delta.shape == (len(gamma) - 1,)
assert np.all(np.isfinite(delta))
assert np.all(delta > 0)

print("zeros    :", len(gamma))
print("spacings :", len(delta))
print("first    :", delta[:10])
print("gamma:")
print(describe(gamma))
print("delta:")
print(describe(delta))


## 02 — Describe

In [ ]:
gamma_range = gamma[-1] - gamma[0]
mean_spacing = np.mean(delta)
range_per_spacing = gamma_range / len(delta)

print("range:", gamma_range)
print("mean spacing:", mean_spacing)
print("range / number of spacings:", range_per_spacing)
assert np.isclose(mean_spacing, range_per_spacing)

percentile_levels = [0, 1, 5, 25, 50, 75, 95, 99, 100]
for p, value in zip(percentile_levels, np.percentile(delta, percentile_levels)):
    print(f"{p:>3}% : {value:.12f}")

BLOCK_SIZE = 1000
num_blocks = len(delta) // BLOCK_SIZE
block_means = np.array([
    np.mean(delta[i * BLOCK_SIZE:(i + 1) * BLOCK_SIZE])
    for i in range(num_blocks)
])
remainder_delta = delta[num_blocks * BLOCK_SIZE:]
print("block size:", BLOCK_SIZE)
print("full blocks:", num_blocks)
print("remainder:", len(remainder_delta))
if len(block_means):
    print("local mean min/max/std:", block_means.min(), block_means.max(), block_means.std())

predicted_global = gamma[0] + np.arange(len(gamma)) * mean_spacing
residual_global = gamma - predicted_global
print("global baseline residual std:", np.std(residual_global))


## 03 — Unfold

Uses the same 1000-spacing local block baseline defined in 02.

**Critical artifact handling:** if `data/derived/unfolded_spacings.float64` already exists, it is loaded and verified against the manifest. It is **not** overwritten. A hash mismatch is a hard error and stops the pipeline. If the artifact is absent, it is created, verified, and its manifest entry is written.

In [ ]:
UNFOLDED_FILE = DERIVED_DIR / "unfolded_spacings.float64"
derived = manifest.setdefault("derived", {})
unfolded_spec = derived.get("unfolded_spacings")

if UNFOLDED_FILE.exists():
    assert unfolded_spec is not None, (
        f"Existing artifact has no manifest entry: {UNFOLDED_FILE}"
    )
    expected_bytes = unfolded_spec["bytes"]
    expected_sha256 = unfolded_spec["sha256"]
    actual_bytes = UNFOLDED_FILE.stat().st_size
    actual_sha256 = sha256(UNFOLDED_FILE)
    assert actual_bytes == expected_bytes, (
        f"Existing unfolded artifact size mismatch: {actual_bytes} != {expected_bytes}"
    )
    assert actual_sha256 == expected_sha256, (
        f"Existing unfolded artifact SHA-256 mismatch: {actual_sha256} != {expected_sha256}"
    )
    unfolded = np.fromfile(UNFOLDED_FILE, dtype=np.float64)
    assert unfolded.size == len(delta)
    assert np.all(np.isfinite(unfolded))
    assert np.all(unfolded > 0)
    print(f"Verified existing derived artifact: {UNFOLDED_FILE}")
    print("bytes:", actual_bytes)
    print("SHA-256:", actual_sha256)
else:
    local_mean = np.empty_like(delta)
    for i in range(num_blocks):
        start = i * BLOCK_SIZE
        end = start + BLOCK_SIZE
        local_mean[start:end] = block_means[i]
    remainder_start = num_blocks * BLOCK_SIZE
    if remainder_start < len(delta):
        local_mean[remainder_start:] = np.mean(delta[remainder_start:])

    unfolded = (delta / local_mean).astype(np.float64, copy=False)
    assert unfolded.shape == delta.shape
    assert np.all(np.isfinite(unfolded))
    assert np.all(unfolded > 0)
    unfolded.tofile(UNFOLDED_FILE)
    assert UNFOLDED_FILE.exists(), f"Output was not created: {UNFOLDED_FILE}"
    actual_bytes = UNFOLDED_FILE.stat().st_size
    actual_sha256 = sha256(UNFOLDED_FILE)
    expected_bytes = unfolded.size * np.dtype(np.float64).itemsize
    assert actual_bytes == expected_bytes, (
        f"Created unfolded artifact size mismatch: {actual_bytes} != {expected_bytes}"
    )

    manifest["derived"]["unfolded_spacings"] = {
        "source_notebook": "notebooks/03_unfold.ipynb",
        "local_file": str(UNFOLDED_FILE),
        "records": int(unfolded.size),
        "representation": "binary IEEE-754 float64",
        "quantity": "unfolded nearest-neighbor spacings",
        "dtype": "float64",
        "bytes": int(actual_bytes),
        "sha256": actual_sha256,
    }
    with MANIFEST_FILE.open("w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
        f.write("\n")
    print(f"Created and verified derived artifact: {UNFOLDED_FILE}")
    print("bytes:", actual_bytes)
    print("SHA-256:", actual_sha256)

print("unfolded:", len(unfolded))
print("mean:", np.mean(unfolded))
print("std :", np.std(unfolded))


## 04 — Surrogates

No reacquisition or reunfolding here. Surrogates are generated directly from the verified 03 artifact.

In [ ]:
SEED = 20260831
rng = np.random.default_rng(SEED)

surrogate_shuffle = unfolded.copy()
rng.shuffle(surrogate_shuffle)
assert np.array_equal(np.sort(surrogate_shuffle), np.sort(unfolded))

surrogate_iid = rng.choice(unfolded, size=len(unfolded), replace=True)
assert surrogate_iid.shape == unfolded.shape
assert np.all(np.isfinite(surrogate_iid))
assert np.all(surrogate_iid > 0)

surrogate_uniform = rng.uniform(0.0, 2.0, size=len(unfolded))
assert surrogate_uniform.shape == unfolded.shape
assert np.all(np.isfinite(surrogate_uniform))
assert np.all(surrogate_uniform >= 0)

datasets = {
    "observed": unfolded,
    "shuffle": surrogate_shuffle,
    "iid": surrogate_iid,
    "uniform": surrogate_uniform,
}
for name, values in datasets.items():
    print(f"{name:>8}: mean={np.mean(values):.6f} std={np.std(values):.6f} min={np.min(values):.6f} max={np.max(values):.6f}")


## 05 — Compare

Comparison only: no acquisition, unfolding, or surrogate generation. The observed series comes from the verified on-disk artifact from 03.

In [ ]:
required = {
    "shuffle": surrogate_shuffle,
    "iid": surrogate_iid,
    "uniform": surrogate_uniform,
}
datasets = {"observed": unfolded, **required}

for name, values in datasets.items():
    assert values.ndim == 1
    assert len(values) == len(unfolded)
    assert np.all(np.isfinite(values))
    assert np.all(values >= 0)

print("datasets:", ", ".join(datasets))
print("N:", len(unfolded))


In [ ]:
def lag1_correlation(values):
    return np.corrcoef(values[:-1], values[1:])[0, 1]

def block_means_for(values, block_size=BLOCK_SIZE):
    n = len(values) // block_size
    return np.asarray([
        np.mean(values[i * block_size:(i + 1) * block_size])
        for i in range(n)
    ])

print("=== distribution summary ===")
for name, values in datasets.items():
    print(f"{name:>8}: mean={np.mean(values):.8f} std={np.std(values):.8f} min={np.min(values):.8f} max={np.max(values):.8f}")

print("=== lag-1 correlation ===")
for name, values in datasets.items():
    print(f"{name:>8}: {lag1_correlation(values):+.8f}")

print("=== block-mean variation ===")
for name, values in datasets.items():
    means = block_means_for(values)
    if len(means):
        print(f"{name:>8}: blocks={len(means)} min={means.min():.8f} max={means.max():.8f} std={means.std():.8f}")
